# Degree Days & Climate Correction

This example shows how to

1. compute Swiss **heating degree days HGT 20/12** and **accumulated temperature differences ATD** from an outdoor temperature series,
2. climate-correct a measured heating consumption with a **reference climatology**,
3. fit a **fixed-base degree-day regression** (energy vs. outside temperature) and
4. apply the **GT&S method** of the BFE (degree days *and* solar radiation).

All plots are **interactive** — hover to see exact values.

In [ ]:
# Configure Plotly to render as HTML (required for Jupyter Book)
import plotly.io as pio
pio.renderers.default = "notebook_connected"

## Heating degree days from hourly data

The bundled sample contains hourly outdoor temperatures of Zürich-Kloten
for 2023. `heating_degree_days` resamples to daily means, counts a day as
heating day when the daily mean is at most 12 °C and sums `20 °C − T_mean`
over those days (SIA 381/3 convention).

In [ ]:
import pandas as pd
from importlib import resources

from pyedautils.degree_days import heating_degree_days

data_path = resources.files("pyedautils") / "data" / "kloten_weather_sample.csv"
weather = pd.read_csv(data_path, parse_dates=["timestamp"])
temp = weather[["timestamp", "temperature"]]

hgt_monthly = heating_degree_days(temp, freq="MS")
hgt_monthly.round(1)

`n_days` are the calendar days of each month, `n_valid` the days with data,
`completeness` their ratio. Use `freq="YS"` for calendar years or
`freq="season"` for heating seasons (July–June).

In [ ]:
heating_degree_days(temp, freq="YS").round(1)

### Plot against a reference

A reference climatology is usually built from a long series (see
`reference_degree_days` and the open-data downloaders). Here we use
illustrative monthly reference values for the Swiss Plateau.

In [ ]:
from pyedautils.plots import plot_degree_days

# Illustrative HGT 20/12 climatology [Kd] per calendar month (index 1..12)
ref_monthly = pd.Series([560, 470, 390, 240, 90, 15, 2, 3, 70, 230, 420, 530],
                        index=range(1, 13))

fig = plot_degree_days(hgt_monthly, reference=ref_monthly,
                       title="Heating Degree Days 20/12 — Kloten 2023")
fig.show()

## Accumulated temperature difference (SIA 380)

SIA 380:2015 recommends the ATD with a single base temperature, the heating
limit of the building. The default is 12 °C; a building-specific value can
come from the energy signature (`compute_pes(...).tb`).

In [ ]:
from pyedautils.degree_days import accumulated_temperature_difference

atd_monthly = accumulated_temperature_difference(temp, base=12.0)
atd_monthly[["atd", "ht"]].round(1).T

## Climate correction of a measured consumption

The classic correction scales only the weather-dependent share of the
consumption with `DD_ref / DD_actual`; domestic hot water stays untouched.

In [ ]:
from pyedautils.degree_days import climate_correction_factor, normalize_consumption

hgt_2023 = hgt_monthly["hgt"].sum()
hgt_ref = ref_monthly.sum()
factor = climate_correction_factor(hgt_2023, hgt_ref)
print(f"HGT 2023: {hgt_2023:.0f} Kd, reference: {hgt_ref:.0f} Kd, factor: {factor:.3f}")

# Measured 2023 consumption of 120 MWh with 75 % space heating share
e_norm = normalize_consumption(120_000, hgt_2023, hgt_ref, weather_dependent_share=0.75)
print(f"Climate-corrected consumption: {e_norm:,.0f} kWh")

Monthly factors follow from a monthly series and the climatology (matched by calendar month):

In [ ]:
climate_correction_factor(hgt_monthly["hgt"], ref_monthly).round(2)

## Degree-day regression with a fixed base temperature

`fit_degree_day_line` fits `E = a + b · HDD(base)` and reports the ASHRAE
Guideline 14 metrics CV(RMSE) and NMBE. The base temperature is not
searched; use the balance temperature of the energy signature or a known
heating limit.

In [ ]:
from pyedautils.degree_days import fit_degree_day_line
from pyedautils.plots import plot_degree_day_regression

sig_path = resources.files("pyedautils") / "data" / "bldg_engy_sig_simple.csv"
sig = pd.read_csv(sig_path, sep=";", parse_dates=["timestamp"])
energy = sig[["timestamp", "value"]]
temperature = sig[["timestamp", "temperature"]]

line = fit_degree_day_line(energy, temperature, base=15.0)
print(f"intercept = {line.intercept:.1f} kWh/d, slope = {line.slope:.2f} kWh/Kd, "
      f"R² = {line.r2:.3f}, CV(RMSE) = {line.cvrmse:.1f} %, NMBE = {line.nmbe:.2f} %")

fig = plot_degree_day_regression(energy, temperature, base=15.0)
fig.show()

## GT&S method (BFE / Prognos)

The degree-day and radiation method weights monthly deviations of degree
days (base 20 °C, no heating limit) and global radiation with functional
elasticities and the normal-year load curve. The documented parameters for
multi-family houses in 2010 reproduce the published annual factor of 1.063
for the Swiss weather of 2010.

In [ ]:
from pyedautils.degree_days import gts_correction_factor, GTS_MFH_2010_EXAMPLE, GTS_CH_2010_WEATHER

ex = GTS_MFH_2010_EXAMPLE
res = gts_correction_factor(
    GTS_CH_2010_WEATHER["gt"], ex["gt_norm"],
    GTS_CH_2010_WEATHER["rad"], ex["rad_norm"],
    ex["load_share_norm"], ex["elasticity_gt"], ex["elasticity_rad"],
)
print(f"ratio = {res.ratio:.3f} (divide the consumption), factor = {res.factor:.3f} (multiply)")
res.monthly.round(3)